In [1]:
import sys, tensorflow as tf, torch, transformers, sklearn
print(sys.version)
print("TF", tf.__version__, tf.config.list_physical_devices("GPU"))
print("Torch MPS:", torch.backends.mps.is_available())
print("transformers", transformers.__version__, "| sklearn", sklearn.__version__)

/Users/naren/Projects/Intelligent_Customer_Support_Ticket_System_Final_Project/Intelligent_Customer_Support_Ticket_System_Final_Project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


3.11.8 (v3.11.8:db85d51d3e, Feb  6 2024, 18:02:37) [Clang 13.0.0 (clang-1300.0.29.30)]
TF 2.21.0 []
Torch MPS: True
transformers 5.20.2 | sklearn 1.9.1


In [3]:
import sys
print(sys.executable)
print(sys.version)
%pip list | grep -i -E "tensorflow|torch|transformers|scikit|pandas"


/usr/local/bin/python3
3.11.8 (v3.11.8:db85d51d3e, Feb  6 2024, 18:02:37) [Clang 13.0.0 (clang-1300.0.29.30)]
pandas                    3.0.5
scikit-learn              1.9.0

[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install tensorflow==2.16.2 tensorflow-metal
%pip install -r ../requirements.txt

  Using cached tensorflow-2.16.2-cp311-cp311-macosx_12_0_arm64.whl.metadata (4.1 kB)
  Using cached ml_dtypes-0.3.2-cp311-cp311-macosx_10_9_universal2.whl.metadata (20 kB)
  Using cached protobuf-4.25.9-cp37-abi3-macosx_10_9_universal2.whl.metadata (541 bytes)
  Using cached tensorboard-2.16.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached tensorflow_io_gcs_filesystem-0.37.1-cp311-cp311-macosx_12_0_arm64.whl.metadata (14 kB)
  Using cached numpy-1.26.4-cp311-cp311-macosx_11_0_arm64.whl.metadata (114 kB)
  Using cached markdown-3.11.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-any.whl.metadata (1.1 kB)
Using cached tensorflow-2.16.2-cp311-cp311-macosx_12_0_arm64.whl (227.0 MB)
Using cached ml_dtypes-0.3.2-cp311-cp311-macosx_10_9_universal2.whl (389 kB)
Using cached numpy-1.26.4-cp311-cp311-macosx_11_0_arm64.whl (14.0 MB)
Using cached protobuf-4.25.9-cp37-abi3-macosx_10_9_universal2.whl (394 kB)
Using cached tensorboard-2.16.2-py3-none-any

In [1]:
   %pip install -r ../requirements.txt


[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip install tensorflow-metal==1.1.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 4.9 MB/s eta 0:00:00ta 0:00:01
  Attempting uninstall: tensorflow-metal
    Found existing installation: tensorflow-metal 1.2.0
    Uninstalling tensorflow-metal-1.2.0:
      Successfully uninstalled tensorflow-metal-1.2.0

[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip uninstall -y tensorflow-metal

Found existing installation: tensorflow-metal 1.1.0
Uninstalling tensorflow-metal-1.1.0:
  Successfully uninstalled tensorflow-metal-1.1.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw/twcs.csv")

df = pd.read_csv(
    RAW,
    dtype={"tweet_id": "int64", "author_id": "string", "text": "string",
           "response_tweet_id": "string", "in_response_to_tweet_id": "float64"},
    encoding="utf-8",
)
df["created_at"] = pd.to_datetime(df["created_at"], format="%a %b %d %H:%M:%S %z %Y")

print(df.shape)
print(df.dtypes, "\n")
print("Inbound split:\n", df["inbound"].value_counts(), "\n")
print("Nulls:\n", df.isna().sum(), "\n")
print("Date range:", df["created_at"].min(), "→", df["created_at"].max(), "\n")
print("Top 10 company handles:\n",
      df.loc[~df["inbound"], "author_id"].value_counts().head(10))
df.head(3)

(2811774, 7)
tweet_id                                 int64
author_id                               string
inbound                                   bool
created_at                 datetime64[us, UTC]
text                                    string
response_tweet_id                       string
in_response_to_tweet_id                float64
dtype: object 

Inbound split:
 inbound
True     1537843
False    1273931
Name: count, dtype: int64 

Nulls:
 tweet_id                         0
author_id                        0
inbound                          0
created_at                       0
text                             0
response_tweet_id          1040629
in_response_to_tweet_id     794335
dtype: int64 

Date range: 2008-05-08 20:13:59+00:00 → 2017-12-03 23:14:01+00:00 

Top 10 company handles:
 author_id
AmazonHelp         169840
AppleSupport       106860
Uber_Support        56270
SpotifyCares        43265
Delta               42253
Tesco               38573
AmericanAir         36764
TMo

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,2017-10-31 22:10:47+00:00,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,2017-10-31 22:11:45+00:00,@sprintcare and how do you propose we do that,<NA>,1.0
2,3,115712,True,2017-10-31 22:08:27+00:00,@sprintcare I have sent several private messag...,1,4.0


In [3]:
# 1. Year distribution (spot the stray old rows)
print(df["created_at"].dt.year.value_counts().sort_index(), "\n")

# 2. Tickets = inbound tweets that START a conversation
tickets = df[df["inbound"] & df["in_response_to_tweet_id"].isna()].copy()
print("First-contact tickets:", len(tickets))

# 3. Attach the company that replied (first id in response_tweet_id, outbound only)
company_map = df.loc[~df["inbound"]].set_index("tweet_id")["author_id"]
first_resp = pd.to_numeric(tickets["response_tweet_id"].str.split(",").str[0],
                           errors="coerce").astype("Int64")
tickets["company"] = first_resp.map(company_map)

print("Share with a company reply:", round(tickets["company"].notna().mean(), 3))
print("\nTop companies among tickets:\n", tickets["company"].value_counts().head(10))

created_at
2008          2
2010          7
2011         11
2012         56
2013         86
2014        199
2015        427
2016       1512
2017    2809474
Name: count, dtype: int64 

First-contact tickets: 787346
Share with a company reply: 0.907

Top companies among tickets:
 company
AmazonHelp      75889
AppleSupport    71124
Uber_Support    38825
SpotifyCares    25767
Delta           23877
AmericanAir     23493
comcastcares    21391
SouthwestAir    19938
TMobileHelp     19581
Ask_Spectrum    16269
Name: count, dtype: int64[pyarrow]


In [ ]:
import sys
sys.path.insert(0, "..")
from lingua import LanguageDetectorBuilder
from src.cleaning import clean_text, URL_RE, MENTION_RE

# 2017 tickets that received a company reply
answered = tickets[(tickets["created_at"].dt.year == 2017) & tickets["company"].notna()].copy()
answered["reply_tweet_id"] = first_resp.loc[answered.index]

# Oversample, then detect language on text with URLs/mentions removed
pool = answered.sample(n=130_000, random_state=42)
stripped = (pool["text"].astype(str)
            .str.replace(URL_RE, " ", regex=True)
            .str.replace(MENTION_RE, " ", regex=True))

detector = LanguageDetectorBuilder.from_all_languages().with_low_accuracy_mode().build()
langs = detector.detect_languages_in_parallel_of(stripped.tolist())
pool["lang"] = [l.iso_code_639_1.name if l else "UNK" for l in langs]
print("Language mix:\n", pool["lang"].value_counts().head(8), "\n")

# Keep English, clean, drop very short tweets
eng = pool[pool["lang"] == "EN"].copy()
eng["text_clean"] = eng["text"].astype(str).map(clean_text)
eng["n_words"] = eng["text_clean"].str.split().str.len()
eng = eng[eng["n_words"] >= 3]
print("English usable tweets:", len(eng))

sample = eng.sample(n=min(100_000, len(eng)), random_state=42)

cols = ["tweet_id", "company", "created_at", "reply_tweet_id", "text", "text_clean", "n_words"]
out = Path("../data/processed/tickets_sample.parquet")
sample[cols].to_parquet(out, index=False)
print("Saved:", sample.shape, "→", out)

for raw, clean in sample[["text", "text_clean"]].head(5).values:
    print(f"\nRAW:   {raw}\nCLEAN: {clean}")

Language mix:
 lang
EN     118062
ES       1342
JA       1275
DE        985
FR        907
UNK       584
PT        568
TL        526
Name: count, dtype: int64 

English usable tweets: 117807
Saved: (100000, 12) → ../data/processed/tickets_sample.parquet

RAW:   @116136 this company charges its customers $50/month for this internet offer. https://t.co/aiW2O9mE6r
CLEAN: this company charges its customers 50 month for this internet offer

RAW:   .@SCsupport For some reason I am not able to upgrade acct. I have made repeated attempts, none go through. Can you help?
CLEAN: for some reason i am not able to upgrade account i have made repeated attempts none go through can you help

RAW:   @JetBlue I was able to book the first leg of my trip-do you know when sept 9th 2018 will be available?
CLEAN: i was able to book the first leg of my trip do you know when sept 9th 2018 will be available

RAW:   @Delta when will you start enforcing the size and quantity of carry on bags. Charge people for carr

In [6]:
%pip install lingua-language-detector

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.7/96.7 MB 4.2 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
